# NeuroGraph HCPTask Connectome Classification with GNNVisualizer

This notebook trains a graph-level GraphSAGE model on PyTorch Geometric `NeuroGraphDataset(name="HCPTask")`, then renders a full 400-node brain connectome graph with `GNNVisualizer`.

HCPTask is a brain connectomics graph classification task. PyG reports 7,443 graphs and 7 graph classes. Each graph has 400 nodes in the current PyG release, which makes it a clean fixed-size stress test for larger GNN model visualization.

Source docs: [PyG NeuroGraphDataset](https://pytorch-geometric.readthedocs.io/en/stable/generated/torch_geometric.datasets.NeuroGraphDataset.html) and [NeuroGraph documentation](https://neurograph.readthedocs.io/).

If imports fail in a fresh kernel, install the runtime packages first:

```bash
python3 -m pip install torch torch-geometric
```

Optional environment variables: `HCP_TASK_EPOCHS`, `HCP_TASK_MAX_TRAIN_GRAPHS`, and `HCP_TASK_HIDDEN_CHANNELS`.

The first download can be large because PyG retrieves the full HCPTask archive.

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display
from torch_geometric.datasets import NeuroGraphDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool

from gnn_exp import GNNVisualizer

In [ ]:
SEED = 7
torch.manual_seed(SEED)

EPOCHS = int(os.environ.get("HCP_TASK_EPOCHS", "3"))
MAX_TRAIN_GRAPHS = int(os.environ.get("HCP_TASK_MAX_TRAIN_GRAPHS", "192"))
HIDDEN_CHANNELS = int(os.environ.get("HCP_TASK_HIDDEN_CHANNELS", "16"))
BATCH_SIZE = 16


def compact_connectome_features(data):
    x = data.x.float()
    row_mean = x.mean(dim=1, keepdim=True)
    row_std = x.std(dim=1, keepdim=True, unbiased=False)
    row_abs = x.abs().mean(dim=1, keepdim=True)
    row_pos = x.clamp_min(0).mean(dim=1, keepdim=True)
    row_neg = (-x.clamp_max(0)).mean(dim=1, keepdim=True)
    degree = torch.bincount(data.edge_index[0], minlength=data.num_nodes).float().view(-1, 1)
    degree = degree / degree.max().clamp_min(1.0)
    node_index = torch.linspace(0, 1, data.num_nodes).view(-1, 1)
    node_phase = torch.cat([
        torch.sin(node_index * torch.pi),
        torch.cos(node_index * torch.pi),
    ], dim=1)
    return torch.cat([row_mean, row_std, row_abs, row_pos, row_neg, degree, node_phase], dim=1)


def prepare_graph(data):
    data = data.clone()
    data.x = compact_connectome_features(data)
    data.y = data.y.view(-1).long()
    return data


def split_graphs(graphs, train_fraction=0.8):
    count = len(graphs)
    order = torch.randperm(count).tolist()
    split = max(1, min(count - 1, int(count * train_fraction)))
    train_graphs = [graphs[index] for index in order[:split]]
    test_graphs = [graphs[index] for index in order[split:]]
    return train_graphs, test_graphs


dataset = NeuroGraphDataset(root=str(repo_root / "data" / "neurograph"), name="HCPTask")
train_count = min(len(dataset), MAX_TRAIN_GRAPHS)
all_graphs = [prepare_graph(dataset[index]) for index in range(train_count)]
train_graphs, test_graphs = split_graphs(all_graphs)
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE)
visual_data = prepare_graph(dataset[0])
query_pair = visual_data.edge_index[:, 0].tolist()

num_features = visual_data.x.size(1)
num_classes = dataset.num_classes

display(Markdown(
    f"Loaded **HCPTask** with {len(dataset)} graphs and {num_classes} classes. "
    f"The compact visible node feature matrix has {num_features} columns. "
    f"Visual graph: {visual_data.num_nodes} nodes, {visual_data.edge_index.size(1)} directed edges."
))

In [ ]:
class HCPTaskGraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.act1 = nn.Tanh()
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        x = self.act1(self.conv1(x, edge_index))
        x = self.act2(self.conv2(x, edge_index))
        graph_embedding = global_mean_pool(x, batch)
        return self.classifier(graph_embedding)

In [ ]:
def accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            logits = model(batch.x, batch.edge_index, batch.batch)
            pred = logits.argmax(dim=1)
            target = batch.y.view(-1).long()
            correct += int((pred == target).sum())
            total += int(target.numel())
    return correct / max(total, 1)


def train_model(model, loader, epochs=EPOCHS):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.006, weight_decay=5e-4)
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for batch in loader:
            optimizer.zero_grad()
            logits = model(batch.x, batch.edge_index, batch.batch)
            loss = F.cross_entropy(logits, batch.y.view(-1).long())
            loss.backward()
            optimizer.step()
            total_loss += float(loss.detach()) * batch.num_graphs
        if epoch == 1 or epoch == epochs:
            avg_loss = total_loss / max(len(loader.dataset), 1)
            display(Markdown(f"Epoch {epoch}: train loss {avg_loss:.4f}"))
    return model


model = HCPTaskGraphSAGE(num_features, HIDDEN_CHANNELS, num_classes)
model = train_model(model, train_loader)
test_acc = accuracy(model, test_loader)
display(Markdown(f"Held-out accuracy on the small demo split: **{test_acc:.3f}**"))

The next cell builds the widget for the full 400-node connectome. `renderer="auto"` prefers WebGPU/WebGL when the browser supports it.

In [ ]:
visualizer = GNNVisualizer(viewportHeight=1120, autoFit=True)
visualizer.add_model(
    data=visual_data,
    model=model.eval(),
    subgraphSample=False,
    queries=[query_pair],
    mode="graph",
)

assert visualizer.renderer == "auto"
assert visualizer.autoFit is True
assert visualizer.viewportHeight == 1120
assert visualizer.modelInfo["conv1"]["type"] == "SAGEConv"
assert visualizer.modelInfo["conv1"].get("aggregation") == "mean"
assert len(visualizer.graphData["x"]) == visual_data.num_nodes
assert "graphAggregation" in visualizer.intmData
assert len(visualizer.intmData["act1"][0]) == HIDDEN_CHANNELS

display(Markdown(
    "| Captured object | Value |\n"
    "| --- | ---: |\n"
    f"| Nodes | {len(visualizer.graphData['x'])} |\n"
    f"| Edges | {visual_data.edge_index.size(1)} |\n"
    f"| Hidden channels | {HIDDEN_CHANNELS} |\n"
    f"| Viewport height | {visualizer.viewportHeight}px |"
))

In [ ]:
display(visualizer)